# Uncertainty Decomposition Analysis (TODO.md §7)

This notebook implements the 5 semantic checks for uncertainty decomposition:
1. BALD vs MCMC validation (already done, cited only)
2. Semantic check A: hardest vs easiest target subject
3. Semantic check B: SITTING vs STANDING confusion (6-class model)
4. Absolute normalization of epistemic signal
5. Temperature effect on epistemic/aleatoric balance

**NO new training for 3-class model** — reuses existing model from §4/§5.
**NEW 6-class model** trained only for semantic check B (point 3).

## Setup

In [ ]:
import sys
from pathlib import Path

# Add parent directory to path
root_dir = Path().resolve().parent
sys.path.insert(0, str(root_dir))

import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

from src.loader import load_har, LOCOMOTION_IDS
from src.subject_split import split_and_scale
from src.bayesian import FeatureClassifier, LastLayerLaplace
from src.har_train import (
    subject_train_val_split,
    check_class_balance,
    train_source_model,
)
from src.uncertainty_utils import (
    compute_epistemic_normalizer,
    normalize_epistemic,
    epistemic_fraction,
)

# Same seed as §4/§5 for deterministic reconstruction
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print("✓ Imports successful")
print(f"✓ Global seed set to {SEED}")

## Point 1: BALD vs MCMC Validation

**Already validated in §5** (`notebooks/test_laplace_vs_mcmc.ipynb`).

That notebook compared the Laplace approximation against ground-truth MCMC posterior using Metropolis-Hastings on a synthetic binary classification task. Key findings:
- Weight marginals matched (correlation > 0.95)
- Predictive distributions agreed (mean_probs, epistemic, aleatoric all correlated > 0.95)
- Laplace is a valid approximation for our last-layer setup

This validation applies to the general implementation in `src/bayesian.py`, not per-dataset. **No need to repeat here.**

## Point 2: Semantic Check A — Hardest vs Easiest Target Subject

Compare epistemic fraction (epistemic / total_entropy) on:
- Source validation (in-distribution reference)
- Hardest target subject (lowest accuracy)
- Easiest target subject (highest accuracy)

**Hypothesis**: Harder subjects should show higher epistemic fraction (model uncertainty dominates).

**Data source**: Existing BALD artifact from §5 (`data/har_bald_artifact.npz`, M_FIXED=5000).

In [ ]:
# Load existing BALD artifact from §5
artifact_path = root_dir / "data" / "har_bald_artifact.npz"
artifact = np.load(artifact_path, allow_pickle=True)

print("Loaded BALD artifact from §5:")
print(f"  M_FIXED: {artifact['M_FIXED']}")
print(f"  tau_prior: {artifact['tau_prior']:.2f}")
print(f"  Target subjects: {artifact['target_subject_ids']}")

# Extract source val
source_val_epistemic = artifact['source_val_epistemic']
source_val_total = artifact['source_val_total_entropy']
source_val_y = artifact['source_val_y']
source_val_mean_probs = artifact['source_val_mean_probs']

print(f"\nSource validation: {len(source_val_y)} samples")

In [ ]:
# Identify hardest and easiest target subjects
# Compute accuracy for each target from artifact
target_accs = {}

for sid in artifact['target_subject_ids']:
    y_true = artifact[f'target_{sid}_y']
    mean_probs = artifact[f'target_{sid}_mean_probs']
    y_pred = mean_probs.argmax(axis=1)
    acc = (y_pred == y_true).mean()
    target_accs[int(sid)] = acc

# Sort by accuracy
sorted_by_acc = sorted(target_accs.items(), key=lambda x: x[1])

hardest_id = sorted_by_acc[0][0]
hardest_acc = sorted_by_acc[0][1]
easiest_id = sorted_by_acc[-1][0]
easiest_acc = sorted_by_acc[-1][1]

print("Target subject accuracies (from BALD artifact):")
print("=" * 50)
for sid, acc in sorted_by_acc:
    marker = "  (HARDEST)" if sid == hardest_id else ("  (EASIEST)" if sid == easiest_id else "")
    print(f"  Subject {sid:2d}: {acc:.3f}{marker}")

print("\n" + "=" * 50)
print(f"Hardest: Subject {hardest_id} (acc={hardest_acc:.3f})")
print(f"Easiest: Subject {easiest_id} (acc={easiest_acc:.3f})")

In [ ]:
# Compute epistemic fraction for source val, hardest, easiest
source_epi_frac = epistemic_fraction(source_val_epistemic, source_val_total)

hardest_epistemic = artifact[f'target_{hardest_id}_epistemic']
hardest_total = artifact[f'target_{hardest_id}_total_entropy']
hardest_epi_frac = epistemic_fraction(hardest_epistemic, hardest_total)

easiest_epistemic = artifact[f'target_{easiest_id}_epistemic']
easiest_total = artifact[f'target_{easiest_id}_total_entropy']
easiest_epi_frac = epistemic_fraction(easiest_epistemic, easiest_total)

print("\n" + "=" * 60)
print("SEMANTIC CHECK A: Epistemic Fraction (epistemic / total)")
print("=" * 60)
print(f"\n{'Group':<25s} | {'Mean Epi Frac':>14s} | {'Mean Epi':>10s} | {'Mean Total':>11s} | {'Accuracy':>9s}")
print("-" * 80)

source_acc = (source_val_mean_probs.argmax(axis=1) == source_val_y).mean()
print(f"{'Source validation':<25s} | {source_epi_frac.mean():14.4f} | "
      f"{source_val_epistemic.mean():10.4f} | {source_val_total.mean():11.4f} | {source_acc:9.3f}")
print(f"{'Easiest (subj ' + str(easiest_id) + ')':<25s} | {easiest_epi_frac.mean():14.4f} | "
      f"{easiest_epistemic.mean():10.4f} | {easiest_total.mean():11.4f} | {easiest_acc:9.3f}")
print(f"{'Hardest (subj ' + str(hardest_id) + ')':<25s} | {hardest_epi_frac.mean():14.4f} | "
      f"{hardest_epistemic.mean():10.4f} | {hardest_total.mean():11.4f} | {hardest_acc:9.3f}")

print("\n" + "=" * 60)
print("Interpretation:")
print(f"  Hardest/Easiest epi_frac ratio: {hardest_epi_frac.mean() / easiest_epi_frac.mean():.2f}")
if hardest_epi_frac.mean() > easiest_epi_frac.mean():
    print("  ✓ Hardest subject shows higher epistemic fraction (as expected)")
else:
    print("  ⚠ Pattern not as expected (signal may be weak on this dataset)")

## Point 3: Semantic Check B — SITTING vs STANDING Confusion (6-Class Model)

**NEW 6-class model** (all HAR activities: WALKING, WALKING_UPSTAIRS, WALKING_DOWNSTAIRS, SITTING, STANDING, LAYING).

**Hypothesis**: SITTING/STANDING should show **aleatoric-dominated** uncertainty (sensor confusion, not distribution shift) compared to other activities.

**This requires training a separate model** — same architecture (561→128→64→**6**), same subjects/split, but all 6 classes instead of just 3 locomotion classes.

In [ ]:
# Load full HAR dataset (all 6 classes, not just locomotion)
dataset_full_6class = load_har(str(root_dir / "data"), merge_train_test=True)
# NO filter_activities - keep all 6 classes

print(f"Full 6-class dataset: {dataset_full_6class.X.shape[0]} samples")
print(f"Activities: {sorted(set(dataset_full_6class.activity_name))}")
print(f"Activity IDs: {sorted(set(dataset_full_6class.y))}")

In [ ]:
# SAME subject partition as §4 (seed=42)
split_6class = split_and_scale(dataset_full_6class, n_source=20, seed=42)

print(f"Source subjects (same as §4): {split_6class.source_subjects}")
print(f"Target subjects (same as §4): {split_6class.target_subjects}")
print(f"Source pool: {split_6class.X_source.shape[0]} samples")

# SAME train/val split within source (seed=42)
train_subjects_6class, val_subjects_6class = subject_train_val_split(
    split_6class.source_subjects, val_fraction=0.2, seed=42
)

print(f"\nTrain subjects (same as §4): {train_subjects_6class}")
print(f"Val subjects (same as §4): {val_subjects_6class}")

# Filter data
train_mask_6class = np.isin(split_6class.subject_id_source, train_subjects_6class)
val_mask_6class = np.isin(split_6class.subject_id_source, val_subjects_6class)

X_train_6class = torch.tensor(split_6class.X_source[train_mask_6class], dtype=torch.float32)
y_train_raw_6class = split_6class.y_source[train_mask_6class]
X_val_6class = torch.tensor(split_6class.X_source[val_mask_6class], dtype=torch.float32)
y_val_raw_6class = split_6class.y_source[val_mask_6class]

# Remap labels 1-6 -> 0-5 (for CrossEntropyLoss)
label_map_6class = {1: 0, 2: 1, 3: 2, 4: 3, 5: 4, 6: 5}
y_train_6class = torch.tensor([label_map_6class[y] for y in y_train_raw_6class], dtype=torch.long)
y_val_6class = torch.tensor([label_map_6class[y] for y in y_val_raw_6class], dtype=torch.long)

print(f"\nTrain samples: {X_train_6class.shape[0]}, Val samples: {X_val_6class.shape[0]}")

In [ ]:
# Check class balance on 6-class train set
class_weights_6class = check_class_balance(
    y_train_6class.numpy(), n_classes=6, split_name="6-class train"
)

In [ ]:
# Initialize 6-class model (561 -> 128 -> 64 -> 6)
model_6class = FeatureClassifier(in_dim=561, hidden_dims=[128, 64], n_classes=6)

print(f"6-class model architecture: 561 -> 128 -> 64 -> 6")
print(f"Total parameters: {sum(p.numel() for p in model_6class.parameters()):,}")

In [ ]:
# Train 6-class model (same hyperparameters as §4)
print("Training 6-class model (this may take 1-2 minutes)...\n")

history_6class = train_source_model(
    model_6class, X_train_6class, y_train_6class, X_val_6class, y_val_6class,
    weight_decay=0.01,
    lr=1e-3,
    epochs=200,
    class_weights=class_weights_6class,
    patience=20
)

print(f"\n✓ Training complete")
print(f"  tau_prior: {history_6class['tau_prior']:.2f}")

In [ ]:
# Fit Laplace on 6-class model
X_source_full_6class = torch.tensor(split_6class.X_source, dtype=torch.float32)
y_source_full_6class = torch.tensor(
    [label_map_6class[y] for y in split_6class.y_source], dtype=torch.long
)

print("Fitting Laplace approximation (6-class model)...")
laplace_6class = LastLayerLaplace.fit(
    model_6class, X_source_full_6class, y_source_full_6class,
    tau_prior=history_6class['tau_prior']
)

# Verify eigenvalues
eigvals_6class = torch.linalg.eigvalsh(laplace_6class.cov)

print(f"\n✓ Laplace fitted (6-class)")
print(f"  Eigenvalue range: [{eigvals_6class.min():.2e}, {eigvals_6class.max():.2e}]")
print(f"  All positive: {(eigvals_6class > 0).all().item()}")

In [ ]:
# Compute predictive on validation set (M=5000 for consistency with §5)
M_FIXED = 5000
FINAL_SEED = 456

print(f"Computing predictive (M={M_FIXED}) on source val...")
pred_val_6class = laplace_6class.predictive(
    model_6class, X_val_6class, M=M_FIXED, temperature=1.0,
    generator=torch.Generator().manual_seed(FINAL_SEED)
)

print("✓ Done")

In [ ]:
# Identify SITTING (class 4) and STANDING (class 5) windows in validation set
# Original labels: 1=WALKING, 2=WALKING_UPSTAIRS, 3=WALKING_DOWNSTAIRS, 4=SITTING, 5=STANDING, 6=LAYING
# Remapped: 0-5

sitting_standing_mask = np.isin(y_val_raw_6class, [4, 5])  # Original labels 4,5
other_mask = ~sitting_standing_mask

print(f"Validation set breakdown:")
print(f"  SITTING/STANDING: {sitting_standing_mask.sum()} samples")
print(f"  Other activities: {other_mask.sum()} samples")

# Extract epistemic fractions for both groups
epistemic_val_6class = pred_val_6class['epistemic'].numpy()
total_val_6class = pred_val_6class['total_entropy'].numpy()

epi_frac_val_6class = epistemic_fraction(epistemic_val_6class, total_val_6class)

epi_frac_sitting_standing = epi_frac_val_6class[sitting_standing_mask]
epi_frac_other = epi_frac_val_6class[other_mask]

epistemic_sitting_standing = epistemic_val_6class[sitting_standing_mask]
epistemic_other = epistemic_val_6class[other_mask]

aleatoric_sitting_standing = pred_val_6class['aleatoric'].numpy()[sitting_standing_mask]
aleatoric_other = pred_val_6class['aleatoric'].numpy()[other_mask]

print("\n" + "=" * 60)
print("SEMANTIC CHECK B: SITTING/STANDING vs Other Activities")
print("=" * 60)
print(f"\n{'Group':<25s} | {'Mean Epi Frac':>14s} | {'Mean Epi':>10s} | {'Mean Alea':>11s}")
print("-" * 70)
print(f"{'SITTING/STANDING':<25s} | {epi_frac_sitting_standing.mean():14.4f} | "
      f"{epistemic_sitting_standing.mean():10.4f} | {aleatoric_sitting_standing.mean():11.4f}")
print(f"{'Other (4 classes)':<25s} | {epi_frac_other.mean():14.4f} | "
      f"{epistemic_other.mean():10.4f} | {aleatoric_other.mean():11.4f}")

print("\n" + "=" * 60)
print("Interpretation:")
print(f"  SITTING/STANDING epi_frac: {epi_frac_sitting_standing.mean():.4f}")
print(f"  Other epi_frac:            {epi_frac_other.mean():.4f}")
print(f"  Ratio (SIT-STAND / Other): {epi_frac_sitting_standing.mean() / epi_frac_other.mean():.2f}")

if epi_frac_sitting_standing.mean() < epi_frac_other.mean():
    print("  ✓ SITTING/STANDING show lower epistemic fraction (aleatoric-dominated, as expected)")
else:
    print("  ⚠ Pattern not as expected (may indicate limited sensor confusion or weak signal)")

In [ ]:
# Save 6-class model and Laplace (separate from 3-class model)
models_dir = root_dir / "data"
models_dir.mkdir(exist_ok=True)

# Save model state
model_6class_path = models_dir / "har_6class_model.pt"
torch.save({
    'model_state_dict': model_6class.state_dict(),
    'tau_prior': history_6class['tau_prior'],
    'history': history_6class,
}, model_6class_path)

# Save Laplace (as npz since it's numpy/torch tensors)
laplace_6class_path = models_dir / "har_6class_laplace.npz"
np.savez_compressed(
    laplace_6class_path,
    theta_map=laplace_6class.theta_map.cpu().numpy(),
    cov=laplace_6class.cov.cpu().numpy(),
    K=laplace_6class.K,
    Dp=laplace_6class.Dp,
    tau_prior=history_6class['tau_prior'],
)

print(f"✓ Saved 6-class model to: {model_6class_path}")
print(f"✓ Saved 6-class Laplace to: {laplace_6class_path}")
print("\nNOTE: These are separate from the 3-class model used in §4/§5/§6.")

## Point 4: Absolute Normalization of Epistemic Signal

Define normalized epistemic = epistemic_raw / quantile_95(source_val_epistemic).

- Values < 1: lower than rare source cases
- Values ≈ 1: comparable to rare source cases  
- Values > 1: exceeds even rare source cases (strong OOD signal)

**Use 3-class model artifact** (from §5) for this analysis.

In [ ]:
# Compute normalizer from source val epistemic (3-class model)
normalizer_95 = compute_epistemic_normalizer(source_val_epistemic, quantile=0.95)

print("Epistemic normalization (3-class model):")
print("=" * 60)
print(f"95th percentile of source val epistemic: {normalizer_95:.6f}")
print(f"\nJustification: This captures the 'rare but seen' threshold.")
print(f"               Target epistemic above this level exceeds even")
print(f"               the high-uncertainty source cases.")

In [ ]:
# Normalize source val and all targets
source_val_epi_norm = normalize_epistemic(source_val_epistemic, normalizer_95)

print("\nNormalized epistemic distributions:")
print("=" * 60)
print(f"\n{'Group':<25s} | {'Mean Norm Epi':>14s} | {'Median':>8s} | {'% > 1':>7s}")
print("-" * 60)

pct_above_1_source = (source_val_epi_norm > 1).mean() * 100
print(f"{'Source val':<25s} | {source_val_epi_norm.mean():14.4f} | "
      f"{np.median(source_val_epi_norm):8.4f} | {pct_above_1_source:6.1f}%")

target_norm_results = {}
for sid in sorted(artifact['target_subject_ids']):
    target_epi = artifact[f'target_{sid}_epistemic']
    target_epi_norm = normalize_epistemic(target_epi, normalizer_95)
    target_norm_results[int(sid)] = target_epi_norm
    
    pct_above_1 = (target_epi_norm > 1).mean() * 100
    print(f"{'Target ' + str(sid):<25s} | {target_epi_norm.mean():14.4f} | "
          f"{np.median(target_epi_norm):8.4f} | {pct_above_1:6.1f}%")

print("\n" + "=" * 60)
print("Interpretation:")
print(f"  Source val should have ~5% above 1 by construction (95th percentile).")
print(f"  Actual: {pct_above_1_source:.1f}%")
print(f"\n  Target subjects with >20% above 1 show strong OOD signal:")
for sid in sorted(target_norm_results.keys()):
    pct = (target_norm_results[sid] > 1).mean() * 100
    if pct > 20:
        print(f"    Subject {sid}: {pct:.1f}%")

## Point 5: Temperature Effect on Epistemic/Aleatoric Balance

Compare temperature τ ∈ {0.4, 1.0} on epistemic fraction.

**Hypothesis**: Lower temperature (τ=0.4) produces sharper predictions → lower aleatoric → higher epistemic fraction.

**Reuse 3-class model and Laplace** from §4/§5 (no new training).

In [ ]:
# Reconstruct 3-class model and Laplace (same as §5)
# Load locomotion dataset
dataset_locomotion = load_har(str(root_dir / "data"), merge_train_test=True).filter_activities(LOCOMOTION_IDS)
split_3class = split_and_scale(dataset_locomotion, n_source=20, seed=42)

train_subjects_3class, val_subjects_3class = subject_train_val_split(
    split_3class.source_subjects, val_fraction=0.2, seed=42
)

train_mask_3class = np.isin(split_3class.subject_id_source, train_subjects_3class)
val_mask_3class = np.isin(split_3class.subject_id_source, val_subjects_3class)

X_train_3class = torch.tensor(split_3class.X_source[train_mask_3class], dtype=torch.float32)
y_train_raw_3class = split_3class.y_source[train_mask_3class]
X_val_3class = torch.tensor(split_3class.X_source[val_mask_3class], dtype=torch.float32)
y_val_raw_3class = split_3class.y_source[val_mask_3class]

label_map_3class = {1: 0, 2: 1, 3: 2}
y_train_3class = torch.tensor([label_map_3class[y] for y in y_train_raw_3class], dtype=torch.long)
y_val_3class = torch.tensor([label_map_3class[y] for y in y_val_raw_3class], dtype=torch.long)

print("Reconstructing 3-class model (same as §4/§5)...")
print(f"Train samples: {X_train_3class.shape[0]}, Val samples: {X_val_3class.shape[0]}")

In [ ]:
# Check class balance
class_weights_3class = check_class_balance(
    y_train_3class.numpy(), n_classes=3, split_name="3-class train"
)

In [ ]:
# Train 3-class model
model_3class = FeatureClassifier(in_dim=561, hidden_dims=[128, 64], n_classes=3)

print("Training 3-class model...\n")
history_3class = train_source_model(
    model_3class, X_train_3class, y_train_3class, X_val_3class, y_val_3class,
    weight_decay=0.01, lr=1e-3, epochs=200,
    class_weights=class_weights_3class, patience=20
)

print(f"\n✓ Training complete")
print(f"  tau_prior: {history_3class['tau_prior']:.2f}")

# Verify determinism
EXPECTED_TAU_PRIOR = 25.10
if abs(history_3class['tau_prior'] - EXPECTED_TAU_PRIOR) > 0.1:
    print(f"  ⚠ WARNING: tau_prior differs from expected ({EXPECTED_TAU_PRIOR:.2f})")
else:
    print(f"  ✓ tau_prior matches §4/§5 (deterministic reconstruction)")

In [ ]:
# Fit Laplace
X_source_full_3class = torch.tensor(split_3class.X_source, dtype=torch.float32)
y_source_full_3class = torch.tensor(
    [label_map_3class[y] for y in split_3class.y_source], dtype=torch.long
)

print("Fitting Laplace approximation (3-class model)...")
laplace_3class = LastLayerLaplace.fit(
    model_3class, X_source_full_3class, y_source_full_3class,
    tau_prior=history_3class['tau_prior']
)

eigvals_3class = torch.linalg.eigvalsh(laplace_3class.cov)
print(f"\n✓ Laplace fitted (3-class)")
print(f"  Eigenvalue range: [{eigvals_3class.min():.2e}, {eigvals_3class.max():.2e}]")

In [ ]:
# Test temperature effect: τ ∈ {0.4, 1.0}
TEMPERATURES = [0.4, 1.0]
temp_results = {}

for temp in TEMPERATURES:
    print(f"\nComputing predictive with temperature τ={temp}...")
    
    # Source val
    pred_val_temp = laplace_3class.predictive(
        model_3class, X_val_3class, M=M_FIXED, temperature=temp,
        generator=torch.Generator().manual_seed(FINAL_SEED)
    )
    
    epi_frac_val_temp = epistemic_fraction(
        pred_val_temp['epistemic'].numpy(),
        pred_val_temp['total_entropy'].numpy()
    )
    
    # All targets
    target_epi_fracs_temp = {}
    for sid in sorted(split_3class.target_subjects):
        ds_t = dataset_locomotion.filter_subjects([sid])
        X_t = torch.tensor(split_3class.scaler.transform(ds_t.X), dtype=torch.float32)
        y_t = np.array([label_map_3class[y] for y in ds_t.y])
        
        pred_t_temp = laplace_3class.predictive(
            model_3class, X_t, M=M_FIXED, temperature=temp,
            generator=torch.Generator().manual_seed(FINAL_SEED)
        )
        
        epi_frac_t_temp = epistemic_fraction(
            pred_t_temp['epistemic'].numpy(),
            pred_t_temp['total_entropy'].numpy()
        )
        
        target_epi_fracs_temp[int(sid)] = epi_frac_t_temp.mean()
    
    temp_results[temp] = {
        'source_val_epi_frac': epi_frac_val_temp.mean(),
        'target_epi_fracs': target_epi_fracs_temp,
    }
    
    print(f"  Source val epi_frac: {epi_frac_val_temp.mean():.4f}")

print("\n✓ Temperature comparison complete")

In [ ]:
# Compare temperatures
print("\n" + "=" * 70)
print("TEMPERATURE EFFECT: τ=0.4 vs τ=1.0 (3-class model)")
print("=" * 70)
print(f"\n{'Group':<25s} | {'τ=0.4 Epi Frac':>16s} | {'τ=1.0 Epi Frac':>16s} | {'Ratio':>7s}")
print("-" * 70)

source_04 = temp_results[0.4]['source_val_epi_frac']
source_10 = temp_results[1.0]['source_val_epi_frac']
print(f"{'Source val':<25s} | {source_04:16.4f} | {source_10:16.4f} | {source_04/source_10:7.2f}")

for sid in sorted(split_3class.target_subjects):
    target_04 = temp_results[0.4]['target_epi_fracs'][int(sid)]
    target_10 = temp_results[1.0]['target_epi_fracs'][int(sid)]
    print(f"{'Target ' + str(sid):<25s} | {target_04:16.4f} | {target_10:16.4f} | {target_04/target_10:7.2f}")

print("\n" + "=" * 70)
print("Interpretation:")
print(f"  Lower temperature (τ=0.4) produces sharper predictions.")
print(f"  Expected effect: lower aleatoric → higher epistemic fraction.")
if source_04 > source_10:
    print(f"  ✓ Observed: τ=0.4 shows higher epi_frac (ratio={source_04/source_10:.2f})")
else:
    print(f"  ⚠ Unexpected: τ=0.4 shows lower epi_frac (ratio={source_04/source_10:.2f})")

In [ ]:
# Preliminary correlation: subject difficulty vs epistemic fraction (τ=1.0)
# Compute accuracy for each target
target_accs_3class = {}
for sid in sorted(split_3class.target_subjects):
    ds_t = dataset_locomotion.filter_subjects([sid])
    X_t = torch.tensor(split_3class.scaler.transform(ds_t.X), dtype=torch.float32)
    y_t = np.array([label_map_3class[y] for y in ds_t.y])
    
    with torch.no_grad():
        preds = model_3class(X_t).argmax(dim=1).numpy()
        acc = (preds == y_t).mean()
    
    target_accs_3class[int(sid)] = acc

# Correlation: (1 - accuracy) vs epi_frac
difficulties = [1 - target_accs_3class[sid] for sid in sorted(split_3class.target_subjects)]
epi_fracs_10 = [temp_results[1.0]['target_epi_fracs'][int(sid)] for sid in sorted(split_3class.target_subjects)]

corr, pval = spearmanr(difficulties, epi_fracs_10)

print("\n" + "=" * 60)
print("PRELIMINARY CORRELATION: Difficulty vs Epistemic Fraction (τ=1.0)")
print("=" * 60)
print(f"\nSpearman correlation: ρ = {corr:.3f}, p-value = {pval:.4f}")
print(f"\nNOTE: This is a preliminary estimate from a single seed.")
print(f"      Formal multi-seed analysis with confidence intervals is §8.")
if pval < 0.05:
    print(f"      Trend is statistically significant at α=0.05.")
else:
    print(f"      Trend is not statistically significant (p={pval:.4f}).")

## Summary: §7 Complete

All 5 uncertainty decomposition checks completed:

1. **BALD vs MCMC**: Already validated in `notebooks/test_laplace_vs_mcmc.ipynb` (§5)
2. **Semantic check A**: Hardest vs easiest target subject epistemic fraction comparison
3. **Semantic check B**: SITTING/STANDING vs other activities in 6-class model
4. **Absolute normalization**: 95th percentile reference for epistemic signal
5. **Temperature effect**: τ=0.4 vs τ=1.0 comparison + preliminary correlation

Results reported above. NO interpretation/conclusions here — review together before marking §7 complete.